# LAB1 - Bài 7 (Titanic): Biến đổi thuộc tính

**Họ tên:** Đinh Công Vinh Quang  
**MSSV:** 6451071061

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

HO_TEN, MSSV = "Đinh Công Vinh Quang", "6451071061"
print(f"{HO_TEN} - {MSSV}")

Đinh Công Vinh Quang - 6451071061


In [2]:
df = pd.read_csv("titanic.csv")

In [3]:
# --- Tiền xử lý (Câu 6) ---
df["deck"] = df["cabin"].str[0]                       # giữ lại ký tự đầu của cabin (boong) trước khi xóa
df = df.drop(columns=["body", "cabin", "boat", "home.dest", "ticket"])

# tuổi: sinh ngẫu nhiên trong [mean - std, mean + std]
rng = np.random.default_rng(42)
mean, std = df["age"].mean(), df["age"].std()
n_null = df["age"].isnull().sum()
df.loc[df["age"].isnull(), "age"] = rng.integers(int(mean - std), int(mean + std) + 1, n_null)

# cổng lên tàu: điền giá trị phổ biến nhất (mode); fare thiếu 1 giá trị: điền trung vị
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])
df["fare"] = df["fare"].fillna(df["fare"].median())
df["deck"] = df["deck"].fillna("U")                   # U = Unknown

In [4]:
# --- Biến đổi thuộc tính (Câu 7) ---
# chia nhóm fare (tính trên giá trị thực, trước khi ép kiểu nguyên)
df["fare_group"] = pd.cut(df["fare"], bins=[-0.01, 7.91, 14.454, 31, 99, 250, np.inf], labels=range(1, 7))
df["fare_group_qcut"] = pd.qcut(df["fare"].rank(method="first"), 6, labels=range(1, 7))   # chia cân bằng bằng qcut

df["fare"] = df["fare"].astype(int)
df["age"] = df["age"].astype(int)

df["title"] = df["name"].str.extract(r",\s*([^\.]+)\.", expand=False).str.strip()
df["title"] = df["title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
common = ["Mr", "Miss", "Mrs", "Master"]
df["title"] = df["title"].where(df["title"].isin(common), "Rare")

df["sex"] = df["sex"].map({"male": 0, "female": 1})
df["embarked"] = df["embarked"].map({"S": 0, "C": 1, "Q": 2})

df["age_group"] = pd.cut(df["age"], bins=[-1, 10, 18, 22, 27, 33, 40, 66, 200], labels=range(1, 9))

In [5]:
print("Tên các danh hiệu (title):"); print(df["title"].value_counts(), "\n")
print("Số hành khách mỗi nhóm tuổi:"); print(df["age_group"].value_counts().sort_index(), "\n")
print("Số hành khách mỗi nhóm giá vé (theo ngưỡng gợi ý):"); print(df["fare_group"].value_counts().sort_index(), "\n")
print("Số hành khách mỗi nhóm giá vé (qcut - cân bằng):"); print(df["fare_group_qcut"].value_counts().sort_index())

Tên các danh hiệu (title):
title
Mr        757
Miss      264
Mrs       198
Master     61
Rare       29
Name: count, dtype: int64 

Số hành khách mỗi nhóm tuổi:
age_group
1     86
2    143
3    170
4    209
5    238
6    218
7    236
8      9
Name: count, dtype: int64 

Số hành khách mỗi nhóm giá vé (theo ngưỡng gợi ý):
fare_group
1    337
2    310
3    332
4    246
5     67
6     17
Name: count, dtype: int64 

Số hành khách mỗi nhóm giá vé (qcut - cân bằng):
fare_group_qcut
1    219
2    218
3    218
4    218
5    218
6    218
Name: count, dtype: int64


In [6]:
df[["name", "title", "sex", "embarked", "age", "age_group", "fare", "fare_group", "fare_group_qcut"]].head(10)

,name,title,sex,embarked,age,age_group,fare,fare_group,fare_group_qcut
0,"Allen, Miss. Elisabeth Walton",Miss,1,0,29,5,211,5,6
1,"Allison, Master. Hudson Trevor",Master,0,0,0,1,151,5,6
2,"Allison, Miss. Helen Loraine",Miss,1,0,2,1,151,5,6
3,"Allison, Mr. Hudson Joshua Creighton",Mr,0,0,30,5,151,5,6
4,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",Mrs,1,0,25,4,151,5,6
5,"Anderson, Mr. Harry",Mr,0,0,48,7,26,3,5
6,"Andrews, Miss. Kornelia Theodosia",Miss,1,0,63,7,77,4,6
7,"Andrews, Mr. Thomas Jr",Mr,0,0,39,6,0,1,1
8,"Appleton, Mrs. Edward Dale (Charlotte Lamson)",Mrs,1,0,53,7,51,4,5
9,"Artagaveytia, Mr. Ramon",Mr,0,1,71,8,49,4,5
